In [1]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
import vectorbt as vbt
import sys

sys.path.insert(0, str(Path('.').resolve().parent))
from dema_strat import load_ohlcv, vbt_frame, free
from constants import CASH, COMMISSION, SESSIONS

TABLE_NAME = "f1_session_chandelier"
SMOOTH_PERIOD = 12

def load_data(file_path, *, verbose=False, use_vwap=False):
    return load_ohlcv(file_path, use_cache=True, add_vwap=use_vwap, verbose=verbose)

def session_mask(index, session):
    start = pd.Timestamp(session["start"]).time()
    end = pd.Timestamp(session["end"]).time()
    tod = np.array(index.to_series().dt.time.values)
    if start <= end:
        return (tod >= start) & (tod < end)
    return (tod >= start) | (tod < end)

def make_signals(data, n1=9, atr_period=14, sl_mult=1.0, use_vwap=False, smooth_period=SMOOTH_PERIOD):
    close = data["Close"]
    high = data["High"]
    low = data["Low"]

    ema1 = vbt.indicators.MA.run(close, window=n1, ewm=True)
    smoothed = vbt.indicators.MA.run(ema1.ma, window=smooth_period, ewm=True)
    atr = vbt.indicators.ATR.run(high, low, close, window=atr_period)

    sl_dist = sl_mult * atr.atr
    sl_pct = sl_dist / close

    long_mask = (smoothed.ma > ema1.ma) & np.isfinite(atr.atr) & (atr.atr > 0)
    short_mask = (ema1.ma > smoothed.ma) & np.isfinite(atr.atr) & (atr.atr > 0)

    if use_vwap:
        long_mask = long_mask & (close > data["VWAP"])
        short_mask = short_mask & (close < data["VWAP"])

    return pd.DataFrame({
        "ema1": ema1.ma,
        "smoothed": smoothed.ma,
        "atr": atr.atr,
        "sl_pct": sl_pct,
        "long": long_mask,
        "short": short_mask,
    })

def build_chandelier_exits(data, signals, trail_mult):
    atr_np = signals["atr"].to_numpy(dtype=np.float64)
    high_np = data["High"].to_numpy(dtype=np.float64)
    low_np  = data["Low"].to_numpy(dtype=np.float64)
    long_in  = signals["long"].to_numpy(dtype=bool)
    short_in = signals["short"].to_numpy(dtype=bool)
    n = len(atr_np)

    long_exits  = np.zeros(n, dtype=bool)
    short_exits = np.zeros(n, dtype=bool)

    peak = np.nan
    ref  = np.nan
    pos  = 0
    for i in range(n):
        if pos == 0:
            if long_in[i] and np.isfinite(atr_np[i]) and atr_np[i] > 0:
                pos  = 1
                peak = high_np[i]
                ref  = atr_np[i]
            elif short_in[i] and np.isfinite(atr_np[i]) and atr_np[i] > 0:
                pos  = -1
                peak = low_np[i]
                ref  = atr_np[i]
            continue
        if pos == 1:
            peak = max(peak, high_np[i])
            trail = peak - trail_mult * ref
            if low_np[i] <= trail:
                long_exits[i] = True
                pos = 0
                peak = np.nan
                ref = np.nan
        else:
            peak = min(peak, low_np[i])
            trail = peak + trail_mult * ref
            if high_np[i] >= trail:
                short_exits[i] = True
                pos = 0
                peak = np.nan
                ref = np.nan

    return long_exits, short_exits

def run_backtest(data, n1=9, atr_period=14, sl_mult=1.0, trail_mult=2.0, use_vwap=False, smooth_period=SMOOTH_PERIOD, session=None):
    signals = make_signals(data, n1=n1, atr_period=atr_period, sl_mult=sl_mult, use_vwap=use_vwap, smooth_period=smooth_period)
    freq = data.index.to_series().diff().median()

    long_signal  = signals["long"].to_numpy(dtype=bool)
    short_signal = signals["short"].to_numpy(dtype=bool)
    close_np = data["Close"].to_numpy(dtype=np.float64)

    n = len(data)
    in_session = np.ones(n, dtype=bool)
    session_end = np.zeros(n, dtype=bool)
    if session is not None:
        in_session = session_mask(data.index, session)
        next_in = np.empty(n, dtype=bool)
        next_in[:-1] = in_session[1:]
        next_in[-1] = False
        session_end = in_session & ~next_in

    long_exits, short_exits = build_chandelier_exits(data, signals, trail_mult)
    long_exits = long_exits | session_end
    short_exits = short_exits | session_end

    long_signal = long_signal & in_session & ~session_end
    short_signal = short_signal & in_session & ~session_end

    portfolio = vbt.Portfolio.from_signals(
        close=close_np,
        entries=long_signal,
        exits=long_exits,
        short_entries=short_signal,
        short_exits=short_exits,
        init_cash=CASH,
        fees=COMMISSION,
        size=100.0,
        size_type="value",
        freq=freq,
        sl_stop=signals["sl_pct"].to_numpy(dtype=np.float64),
        tp_stop=None,
    )

    return portfolio, portfolio.stats(), signals

def run_params(data, n1, atr_period, sl_mult, trail_mult, use_vwap=False, asset=None, timeframe=None, smooth_period=SMOOTH_PERIOD, session=None):
    session_name = session["name"] if session else None
    try:
        portfolio, stats, _ = run_backtest(data, n1=n1, atr_period=atr_period, sl_mult=sl_mult, trail_mult=trail_mult, use_vwap=use_vwap, smooth_period=smooth_period, session=session)
        return {
            "n1": n1, "atr_period": atr_period,
            "sl_mult": sl_mult, "trail_mult": trail_mult,
            "use_vwap": use_vwap,
            "smooth_period": smooth_period,
            "asset": asset,
            "timeframe": timeframe,
            "session": session_name,
            "return_pct": float(stats["Total Return [%]"]) if stats.get("Total Return [%]") is not None else None,
            "max_dd": float(stats["Max Drawdown [%]"]) if stats.get("Max Drawdown [%]") is not None else None,
            "sharpe": float(stats["Sharpe Ratio"]) if stats.get("Sharpe Ratio") is not None else None,
            "win_rate": float(stats["Win Rate [%]"]) if stats.get("Win Rate [%]") is not None else None,
            "profit_factor": float(stats["Profit Factor"]) if stats.get("Profit Factor") is not None else None,
            "expectancy": float(stats["Expectancy"]) if stats.get("Expectancy") is not None else None,
            "trades": int(stats["Total Trades"]) if stats.get("Total Trades") is not None else None,
        }
    except Exception:
        return {
            "n1": n1, "atr_period": atr_period,
            "sl_mult": sl_mult, "trail_mult": trail_mult,
            "use_vwap": use_vwap,
            "smooth_period": smooth_period,
            "asset": asset,
            "timeframe": timeframe,
            "session": session_name,
            "return_pct": None, "max_dd": None, "sharpe": None,
            "win_rate": None, "profit_factor": None,
            "expectancy": None, "trades": None,
        }


In [2]:
file_path = Path("~/market_data/XAUUSD/XAUUSD_M1.csv").expanduser()
data = load_data(file_path, verbose=True)

portfolio, stats, signals = run_backtest(data, trail_mult=2.0, session=SESSIONS[0])
print(stats)

trades = portfolio.trades.records_readable
print(trades)

free(data, signals)

[loader] cache hit: XAUUSD_M1__cache_v1.parquet
Start                                                 0
End                                             2088752
Period                               1450 days 12:33:00
Start Value                                     50000.0
End Value                                  47758.791725
Total Return [%]                              -4.482417
Benchmark Return [%]                         184.185378
Max Gross Exposure [%]                         0.211469
Total Fees Paid                             2286.603255
Max Drawdown [%]                               4.482417
Max Drawdown Duration                1450 days 10:36:00
Total Trades                                      76220
Total Closed Trades                               76220
Total Open Trades                                     0
Open Trade PnL                                      0.0
Win Rate [%]                                   28.41905
Best Trade [%]                                 1.018104


In [ ]:
portfolio.plot()

## Optimization

In [4]:
from itertools import product
import time


db_path = Path("../backtest_results.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")
cursor.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    n1 INTEGER NOT NULL,
    atr_period INTEGER NOT NULL,
    sl_mult REAL NOT NULL,
    trail_mult REAL NOT NULL,
    use_vwap INTEGER NOT NULL,
    smooth_period INTEGER NOT NULL,
    asset TEXT NOT NULL,
    timeframe TEXT NOT NULL,
    session TEXT,
    expectancy REAL,
    win_rate REAL,
    max_dd REAL,
    sharpe_ratio REAL,
    profit_factor REAL,
    num_trades INTEGER,
    return_pct REAL
)
""")
conn.commit()

grid = list(product(
    ["M1", "M5", "M15"], # timeframes
    range(5, 15, 2), # fast ema
    range(6, 16, 2), # smoothing
    [14, 18, 21], # atr period
    [1.0, 2.0, 3.0], # sl mult
    [1.0, 2.0, 3.0], # trail_mult
    [False, True], # use vwap
    SESSIONS, # session windows (closes positions at end)
))

#* Only for test
# grid = list(product(
#     ["M1"], # timeframes
#     [9], # fast ema
#     [12], # smoothing
#     [14], # atr period
#     [1.0], # sl mult
#     [2.0], # trail_mult
#     [True], # use vwap
#     SESSIONS, # sessions
# ))

t0 = time.time()

print(f"[{time.strftime('%H:%M:%S')}] Testing {len(grid)} combinations -> {TABLE_NAME}")
asset = "XAUUSD"
data_by_tf: dict[tuple[str, bool], pd.DataFrame] = {}
for tf in sorted({g[0] for g in grid}):
    for uv in sorted({g[6] for g in grid if g[0] == tf}):
        fp = Path(f"~/market_data/{asset}/{asset}_{tf}.csv").expanduser()
        data_by_tf[(tf, uv)] = load_data(fp, verbose=True, use_vwap=uv)

ONE_LOOP_MEAN_TIME = 0.0
for i, (timeframe, n1, smoothing, ap, sm, tm, uv, session) in enumerate(grid):

    data = data_by_tf[(timeframe, uv)]

    t1 = time.time()
    result = run_params(data, n1, ap, sm, tm, use_vwap=uv, asset=asset, timeframe=timeframe, smooth_period=smoothing, session=session)
    elapsed = time.time() - t1
    cursor.execute(
        f"INSERT INTO {TABLE_NAME} (n1, atr_period, sl_mult, trail_mult, use_vwap, smooth_period, asset, timeframe, session, expectancy, win_rate, max_dd, sharpe_ratio, profit_factor, num_trades, return_pct) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        (result["n1"], result["atr_period"], result["sl_mult"], result["trail_mult"], uv, result["smooth_period"], result["asset"], result["timeframe"], result["session"],
         result["expectancy"], result["win_rate"], result["max_dd"], result["sharpe"], result["profit_factor"], result["trades"], result["return_pct"]),
    )
    conn.commit()
    ONE_LOOP_MEAN_TIME = (elapsed + ONE_LOOP_MEAN_TIME * i) / (i + 1) if i else elapsed
    combinations_left = len(grid) - (i + 1)
    est_left_sec = ONE_LOOP_MEAN_TIME * combinations_left
    if (i + 1) % max(1, len(grid) // 100) == 0 or (i + 1) == len(grid):
        print(
            f"[{time.strftime('%H:%M:%S')}] {i+1:>5}/{len(grid)} tf={timeframe} n1={n1} ap={ap} sm={sm} tm={tm} vwap={uv} sess={session['name']} "
            f"| sharpe={result['sharpe']} ret={result['return_pct']} trades={result['trades']} ({elapsed:.1f}s) "
            f"| mean={ONE_LOOP_MEAN_TIME:.1f}s est_left={est_left_sec/60:.1f}min"
        )

    free()

free(data_by_tf)

sweep_df = pd.read_sql_query(f"SELECT * FROM {TABLE_NAME}", conn)
sweep_df = sweep_df.dropna(subset=["expectancy"])
sweep_df = sweep_df.sort_values("expectancy", ascending=False).reset_index(drop=True)
print(f"Total grid: {len(grid)} | Non-null: {len(sweep_df)}")
sweep_df.head(20)


[10:39:25] Testing 16200 combinations -> f1_session_chandelier
[loader] cache hit: XAUUSD_M1__cache_v1.parquet
[loader] cache hit: XAUUSD_M1__cache_v1.parquet
[loader] cache hit: XAUUSD_M15__cache_v1.parquet
[loader] cache hit: XAUUSD_M15__cache_v1.parquet
[loader] cache hit: XAUUSD_M5__cache_v1.parquet
[loader] cache hit: XAUUSD_M5__cache_v1.parquet
[10:45:19]   162/16200 tf=M1 n1=5 ap=21 sm=1.0 tm=3.0 vwap=False sess=LONDON | sharpe=-78.59385007933804 ret=-8.96861576360419 trades=152497 (2.0s) | mean=2.0s est_left=531.9min
[10:51:11]   324/16200 tf=M1 n1=5 ap=18 sm=2.0 tm=2.0 vwap=False sess=OVERLAP | sharpe=-22.839342614657063 ret=-1.5745758310496145 trades=26113 (2.0s) | mean=2.0s est_left=524.8min
[10:57:01]   486/16200 tf=M1 n1=5 ap=14 sm=3.0 tm=1.0 vwap=True sess=LONDON | sharpe=-19.687779393152496 ret=-1.761981381256674 trades=32050 (1.9s) | mean=2.0s est_left=517.6min
[11:02:40]   648/16200 tf=M1 n1=5 ap=21 sm=3.0 tm=3.0 vwap=True sess=OVERLAP | sharpe=-2.8728793593571327 ret=

,id,n1,atr_period,sl_mult,trail_mult,use_vwap,smooth_period,asset,timeframe,session,expectancy,win_rate,max_dd,sharpe_ratio,profit_factor,num_trades,return_pct
0,12598,7,14,3.0,3.0,1,12,XAUUSD,M15,LONDON,0.001349,37.738483,0.039749,0.074013,1.009357,2149,0.005796
1,12670,7,18,3.0,3.0,1,12,XAUUSD,M15,LONDON,0.001006,37.598152,0.037331,0.056241,1.007032,2165,0.004356
2,14326,11,14,3.0,3.0,1,8,XAUUSD,M15,LONDON,0.000970,37.442076,0.032981,0.053550,1.006758,2158,0.004187
3,14398,11,18,3.0,3.0,1,8,XAUUSD,M15,LONDON,0.000625,37.362132,0.032175,0.035208,1.004392,2176,0.002722
4,13462,9,14,3.0,3.0,1,10,XAUUSD,M15,LONDON,-0.000074,37.436607,0.038020,-0.003994,0.999487,2169,-0.000321
5,12646,7,18,2.0,3.0,1,12,XAUUSD,M15,LONDON,-0.000089,36.528029,0.038256,-0.005088,0.999365,2212,-0.000395
6,12574,7,14,2.0,3.0,1,12,XAUUSD,M15,LONDON,-0.000411,36.645397,0.037705,-0.023335,0.997104,2194,-0.001805
7,14302,11,14,2.0,3.0,1,8,XAUUSD,M15,LONDON,-0.000563,36.479129,0.033370,-0.032084,0.996028,2204,-0.002480
8,14374,11,18,2.0,3.0,1,8,XAUUSD,M15,LONDON,-0.000863,36.359551,0.033085,-0.050319,0.993839,2225,-0.003841
9,13534,9,18,3.0,3.0,1,10,XAUUSD,M15,LONDON,-0.000914,37.328454,0.036331,-0.051423,0.993601,2186,-0.003995


In [ ]:
db_path = Path("../backtest_results.db")
conn = sqlite3.connect(db_path)

vwap_comparison = pd.read_sql_query(
    f"""SELECT use_vwap,
              COUNT(*) AS n,
              AVG(expectancy) AS mean_expectancy,
              AVG(win_rate) AS mean_win_rate,
              AVG(max_dd) AS mean_max_dd,
              AVG(sharpe_ratio) AS mean_sharpe,
              AVG(profit_factor) AS mean_profit_factor,
              AVG(num_trades) AS mean_num_trades,
              AVG(return_pct) AS mean_return_pct
       FROM {TABLE_NAME}
       WHERE sharpe_ratio IS NOT NULL
       GROUP BY use_vwap""",
    conn,
)
print(vwap_comparison)


In [ ]:
session_comparison = pd.read_sql_query(
    """
    SELECT session,
           COUNT(*) AS n,
           AVG(expectancy) AS mean_expectancy,
           AVG(win_rate) AS mean_win_rate,
           AVG(max_dd) AS mean_max_dd,
           AVG(sharpe_ratio) AS mean_sharpe,
           AVG(profit_factor) AS mean_profit_factor,
           AVG(num_trades) AS mean_num_trades,
           AVG(return_pct) AS mean_return_pct
    FROM """ + TABLE_NAME + """
    WHERE session IS NOT NULL AND sharpe_ratio IS NOT NULL AND expectancy >= 0
    GROUP BY session
    ORDER BY mean_expectancy, mean_win_rate
    """,
    conn,
)
print(session_comparison)
